<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/04_Expert/L47_Mixture_of_Experts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L47: Mixture of Experts - Sparse Routing and Expert Specialization

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Expert  
**Lesson:** 47 of 60

---

## Learning Objectives

By the end of this lesson, you will:
- Understand MoE: multiple expert networks and a router that selects top-k
- Implement a minimal MoE layer (router + experts)
- See how MoE reduces active parameters per token

---

## Concept: Mixture of Experts

**MoE**: each layer has many "experts" (e.g. FFNs); a **router** (small network) assigns each token to top-k experts. Only those experts are computed, so total params can be large while **active** params per token stay manageable. Used in Mixtral, GShard, Switch Transformer.

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded.")

## Part 1: Router and Top-K Selection

---

In [ ]:
class Router(nn.Module):
    def __init__(self, d_model, n_experts, top_k=2):
        super().__init__()
        self.linear = nn.Linear(d_model, n_experts)
        self.top_k = top_k

    def forward(self, x):
        logits = self.linear(x)
        top_k_logits, top_k_idx = torch.topk(logits, self.top_k, dim=-1)
        weights = F.softmax(top_k_logits, dim=-1)
        return weights, top_k_idx

router = Router(d_model=256, n_experts=4, top_k=2)
x = torch.randn(2, 10, 256)
w, idx = router(x)
print("Router weights shape:", w.shape, "Top-k indices shape:", idx.shape)

## Part 2: Minimal MoE Layer

---

In [ ]:
class MoELayer(nn.Module):
    def __init__(self, d_model, d_ff, n_experts=4, top_k=2):
        super().__init__()
        self.router = Router(d_model, n_experts, top_k)
        self.experts = nn.ModuleList([nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)) for _ in range(n_experts)])
        self.n_experts = n_experts
        self.top_k = top_k

    def forward(self, x):
        B, T, C = x.shape
        w, idx = self.router(x)
        out = torch.zeros_like(x)
        for k in range(self.top_k):
            expert_weights = w[:, :, k]
            expert_id = idx[:, :, k]
            for e in range(self.n_experts):
                mask = (expert_id == e)
                if mask.any():
                    out[mask] += expert_weights[mask].unsqueeze(-1) * self.experts[e](x[mask])
        return out

moe = MoELayer(256, 512, n_experts=4, top_k=2)
y = moe(x)
print("MoE output shape:", y.shape)

## Part 3: Load a Real MoE Model (Concept)

---

In [ ]:
# Mixtral, etc.: from_pretrained('mistralai/Mixtral-8x7B-v0.1') has 8 experts per layer, top-2.
print("Real MoE models: Mixtral (8 experts, top-2), Switch Transformer. Load with transformers when available.")

## Exercises

1. Vary top_k (1 vs 2) and n_experts (2 vs 8); compare params and speed.
2. Add load balancing loss: encourage uniform usage of experts over batch.
3. Read Mixtral architecture and list differences from dense transformer.

---

## Key Takeaways

1. MoE = router + multiple experts; only top-k experts run per token.
2. Total params grow with experts; active params and compute stay bounded.
3. Load balancing (aux loss or capacity) is needed so experts are used fairly.

---

## Next Lesson

**L48: State Space Models** – S4, Mamba, and alternatives to attention.

---